# Portada con IA para el carrusel de Instagram

Genera **solo la ilustración de fondo de la portada** con Qwen-Image (Nunchaku NVFP4 + Lightning, en la RTX 5090) y compone el carrusel con `src.votaciones.instagram`.

Reglas (D-20 en `docs/DECISIONS.md`):
- La IA **nunca** dibuja datos: ni hemiciclos, ni cifras, ni texto. Todo eso lo pone el código.
- Ilustración **abstracta**: sin personas, sin rostros, sin políticos reales, sin el edificio real del Congreso, sin logos, sin banderas, sin símbolos de partidos y sin colores de partido dominantes.
- La portada y el texto de la publicación dicen «Ilustración de portada generada con IA».

Antes de empezar:
- Ollama sin modelos cargados (`ollama ps`; `ollama stop <modelo>`): hacen falta unos 20 GiB de VRAM libres.
- La primera vez descarga el transformer cuantizado (unos 12–13 GB). Ya hay 7,9 GB a medias en la caché, de la prueba de SRNR, y la descarga continúa desde ahí.
- Lo ejecuta el usuario. Tarda: carga ~1–2 min y cada imagen unos segundos.

In [ ]:
# ===== Configuración =====
import sys
from pathlib import Path

MES = "2026-09"
SEMILLAS = [11, 22, 33]       # tres propuestas para elegir
RANK = 128                    # 128: mejor calidad · 32: algo más rápido
STEPS = 8                     # Lightning v1.1
WIDTH, HEIGHT = 1104, 1472    # 3:4 oficial de Qwen-Image; instagram.py lo recorta a 1080 x 1350

RAIZ = Path.home() / "proyectos/votaciones-congreso"
OUT_DIR = RAIZ / "social" / MES / "ia"
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(RAIZ))

NUNCHAKU_REPO = "nunchaku-ai/nunchaku-qwen-image"
BASE_REPO = "Qwen/Qwen-Image"   # ya en la caché: codificador de texto y VAE

In [ ]:
# ===== 1. GPU libre y nunchaku en FP4 =====
import torch
from nunchaku import NunchakuQwenImageTransformer2DModel
from nunchaku.utils import get_precision

PRECISION = get_precision()
libre, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), "| precisión:", PRECISION,
      f"| VRAM libre {libre/2**30:.1f} de {total/2**30:.1f} GiB")
assert PRECISION == "fp4", "En la 5090 debería ser fp4 (NVFP4)"
if libre / 2**30 < 20:
    print("⚠️  Menos de 20 GiB libres: descarga los modelos de Ollama antes de seguir.")

In [ ]:
# ===== 2. Pesos cuantizados (se descargan una vez) =====
import os
from huggingface_hub import hf_hub_download

# Sin token, Hugging Face limita mucho las descargas grandes. Se reutiliza el de marketmenow.
env_mmn = Path.home() / "proyectos/marketmenow/.env"
if "HF_TOKEN" not in os.environ and env_mmn.exists():
    for linea in env_mmn.read_text().splitlines():
        if linea.startswith("HF_TOKEN="):
            os.environ["HF_TOKEN"] = linea.split("=", 1)[1].strip()
print("HF_TOKEN presente:", bool(os.environ.get("HF_TOKEN")))

version = {4: "lightningv1.0-4steps", 8: "lightningv1.1-8steps"}[STEPS]
ruta_pesos = hf_hub_download(NUNCHAKU_REPO, f"svdq-{PRECISION}_r{RANK}-qwen-image-{version}.safetensors")
print(ruta_pesos)

In [ ]:
# ===== 3. Cargar el pipeline =====
import math, time
from diffusers import FlowMatchEulerDiscreteScheduler, QwenImagePipeline

scheduler = FlowMatchEulerDiscreteScheduler.from_config({
    "base_image_seq_len": 256, "base_shift": math.log(3), "invert_sigmas": False,
    "max_image_seq_len": 8192, "max_shift": math.log(3), "num_train_timesteps": 1000,
    "shift": 1.0, "shift_terminal": None, "stochastic_sampling": False,
    "time_shift_type": "exponential", "use_beta_sigmas": False, "use_dynamic_shifting": True,
    "use_exponential_sigmas": False, "use_karras_sigmas": False,
})
t0 = time.time()
transformer = NunchakuQwenImageTransformer2DModel.from_pretrained(ruta_pesos)
pipe = QwenImagePipeline.from_pretrained(
    BASE_REPO, transformer=transformer, scheduler=scheduler,
    torch_dtype=torch.bfloat16, local_files_only=True,
)
pipe.enable_model_cpu_offload()   # transformer + codificador no caben juntos en 24 GB
print(f"Carga: {time.time() - t0:.0f} s")

## 4. Ilustraciones

El prompt describe un hemiciclo **genérico y vacío**, en los colores del proyecto (verde azulado, arena, naranja suave, carbón), no un lugar ni a nadie reales. Va en inglés porque Qwen-Image lo sigue mejor. El texto de la portada se compone después.

In [ ]:
import json

ESTILO = (
    "Minimalist flat editorial illustration, geometric shapes, soft grain texture, calm and neutral "
    "institutional mood, muted palette of deep teal, warm sand, soft orange accents and charcoal. "
    "No people, no faces, no text, no letters, no numbers, no logos, no flags, no coats of arms, "
    "no party symbols."
)
ESCENA = (
    "Top-down view of an empty generic semicircular assembly hall: concentric curved rows of seats "
    "forming a hemicycle, long soft shadows, morning light coming from one side, the lower half of "
    "the image calm and dark enough to place text over it."
)
PROMPT = f"{ESCENA} {ESTILO}"

registro = []
for semilla in SEMILLAS:
    t0 = time.time()
    imagen = pipe(prompt=PROMPT, width=WIDTH, height=HEIGHT, num_inference_steps=STEPS,
                  true_cfg_scale=1.0, generator=torch.Generator("cpu").manual_seed(semilla)).images[0]
    ruta = OUT_DIR / f"fondo_s{semilla}.png"
    imagen.save(ruta)
    registro.append({"archivo": ruta.name, "semilla": semilla, "prompt": PROMPT, "modelo": BASE_REPO,
                     "pesos": Path(ruta_pesos).name, "pasos": STEPS, "segundos": round(time.time() - t0, 1)})
    print(ruta.name, f"{time.time() - t0:.1f} s")
(OUT_DIR / "generacion.json").write_text(json.dumps(registro, ensure_ascii=False, indent=1), "utf-8")

In [ ]:
# Ver las tres y elegir
from IPython.display import display
from PIL import Image
for r in registro:
    print(r["archivo"]); display(Image.open(OUT_DIR / r["archivo"]).resize((368, 491)))

## 5. Componer el carrusel con la portada elegida

Revisa la imagen antes: si sale texto, una bandera, un logo o algo que recuerde a un partido, prueba otra semilla.

In [ ]:
ELEGIDA = OUT_DIR / f"fondo_s{SEMILLAS[0]}.png"   # cambia a la que prefieras

from src.votaciones import instagram
carpeta = instagram.generar(MES, fondo=ELEGIDA)
print(carpeta)
display(Image.open(carpeta / "01-portada.png").resize((540, 675)))
print((carpeta / "pie.txt").read_text())

In [ ]:
# ===== 6. Liberar la GPU =====
del pipe, transformer
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"VRAM libre: {torch.cuda.mem_get_info()[0]/2**30:.1f} GiB")